# Biohub - Cell Tracking During Development
## Score: .669

## Imports

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

WHEELS_DIR = Path(
    "/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels"
)


def ensure(package, module=None):
    module = module or package
    if importlib.util.find_spec(module) is not None:
        return
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(WHEELS_DIR),
        package,
    ]
    subprocess.check_call(command)


print("wheels:", WHEELS_DIR)
ensure("zarr")
ensure("numcodecs")

In [1]:
import csv
import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import zarr
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

## Configuration

In [2]:
DATA_DIR = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
OUTPUT_PATH = WORK_DIR / "submission.csv"

SCALE_ZYX = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
MAX_LINK_UM = 7.0
SIGMA_ZYX = (1.0, 2.0, 2.0)
MIN_DISTANCE_ZYX = (2, 4, 4)
FIXED_TOP_K = None
RUN_VALIDATION = False

DATA_DIR, OUTPUT_PATH

## Data Loading

In [3]:
def open_volume(path):
    root = zarr.open(str(path), mode="r")
    return root["0"] if "0" in root else root


def read_geff(path):
    graph = zarr.open(str(path), mode="r")
    nodes = graph["nodes"]
    props = nodes["props"]
    return (
        np.asarray(nodes["ids"][:]),
        np.asarray(props["t"]["values"][:]),
        np.asarray(props["z"]["values"][:]),
        np.asarray(props["y"]["values"][:]),
        np.asarray(props["x"]["values"][:]),
        np.asarray(graph["edges"]["ids"][:]),
    )


train_samples = sorted(path.stem for path in TRAIN_DIR.glob("*.zarr"))
test_samples = sorted(path.stem for path in TEST_DIR.glob("*.zarr"))
len(train_samples), len(test_samples), test_samples

(199, 4, ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1'])

## Cell Detection

In [4]:
def local_maxima(smooth, min_distance_zyx=MIN_DISTANCE_ZYX):
    size = tuple(2 * distance + 1 for distance in min_distance_zyx)
    maxima = smooth == ndi.maximum_filter(smooth, size=size)
    maxima[:, (0, -1), :] = False
    maxima[:, :, (0, -1)] = False
    return maxima


def estimate_top_k(frame, percentile=90.0, lower=50, upper=400):
    smooth = ndi.gaussian_filter(frame.astype(np.float32), sigma=SIGMA_ZYX)
    maxima = local_maxima(smooth)
    if not maxima.any():
        return lower
    threshold = np.percentile(smooth, percentile)
    return int(np.clip(np.sum(smooth[maxima] >= threshold), lower, upper))


def refine_peaks(frame, coordinates, radius_zyx=(2, 4, 4)):
    if not len(coordinates):
        return coordinates
    rz, ry, rx = radius_zyx
    z_size, y_size, x_size = frame.shape
    refined = coordinates.copy()
    for index, (z, y, x) in enumerate(coordinates):
        z0, z1 = max(0, z - rz), min(z_size, z + rz + 1)
        y0, y1 = max(0, y - ry), min(y_size, y + ry + 1)
        x0, x1 = max(0, x - rx), min(x_size, x + rx + 1)
        patch = frame[z0:z1, y0:y1, x0:x1]
        offset = np.unravel_index(np.argmax(patch), patch.shape)
        refined[index] = z0 + offset[0], y0 + offset[1], x0 + offset[2]
    return refined


def detect_frame(frame, top_k=None):
    raw = frame.astype(np.float32, copy=False)
    smooth = ndi.gaussian_filter(raw, sigma=SIGMA_ZYX)
    maxima = local_maxima(smooth)
    coordinates = np.argwhere(maxima)
    if not len(coordinates):
        return np.empty((0, 3), dtype=np.int32)
    strengths = smooth[maxima]
    limit = estimate_top_k(raw) if top_k is None else top_k
    coordinates = coordinates[np.argsort(strengths)[::-1][:limit]]
    coordinates = np.unique(refine_peaks(raw, coordinates), axis=0)
    return coordinates.astype(np.int32)

## Cell Linking

In [5]:
@dataclass
class TrackGraph:
    nodes: list[tuple[int, int, int, int, int]]
    edges: list[tuple[int, int]]


def link_frames(source, target, max_distance_um=MAX_LINK_UM):
    if not len(source) or not len(target):
        return []
    source_um = source.astype(np.float64) * SCALE_ZYX
    target_um = target.astype(np.float64) * SCALE_ZYX
    distances_squared = ((source_um[:, None] - target_um[None]) ** 2).sum(axis=2)
    cost = distances_squared.copy()
    cost[distances_squared > max_distance_um**2] = 1e6
    source_indices, target_indices = linear_sum_assignment(cost)
    return [
        (int(i), int(j))
        for i, j in zip(source_indices, target_indices)
        if cost[i, j] < 1e6
    ]


def track_volume(volume, top_k=FIXED_TOP_K, max_distance_um=MAX_LINK_UM):
    nodes, edges = [], []
    previous_coordinates, previous_ids = None, []
    next_id = 1

    for timepoint in range(volume.shape[0]):
        coordinates = detect_frame(np.asarray(volume[timepoint]), top_k=top_k)
        current_ids = list(range(next_id, next_id + len(coordinates)))
        nodes.extend(
            (node_id, timepoint, int(z), int(y), int(x))
            for node_id, (z, y, x) in zip(current_ids, coordinates)
        )
        if previous_coordinates is not None:
            edges.extend(
                (previous_ids[i], current_ids[j])
                for i, j in link_frames(
                    previous_coordinates, coordinates, max_distance_um
                )
            )
        next_id += len(coordinates)
        previous_coordinates, previous_ids = coordinates, current_ids

    return TrackGraph(nodes, edges)

## Local Evaluation

In [6]:
def match_nodes(graph, gt_ids, gt_t, gt_z, gt_y, gt_x):
    predicted = defaultdict(list)
    ground_truth = defaultdict(list)

    for node_id, t, z, y, x in graph.nodes:
        predicted[t].append((node_id, np.array([z, y, x], dtype=np.float64)))
    for node_id, t, z, y, x in zip(gt_ids, gt_t, gt_z, gt_y, gt_x):
        ground_truth[int(t)].append(
            (int(node_id), np.array([z, y, x], dtype=np.float64))
        )

    matches = {}
    for timepoint, pred_nodes in predicted.items():
        gt_nodes = ground_truth.get(timepoint, [])
        if not gt_nodes:
            continue
        pred_coords = np.stack([coords for _, coords in pred_nodes]) * SCALE_ZYX
        gt_coords = np.stack([coords for _, coords in gt_nodes]) * SCALE_ZYX
        distances_squared = ((pred_coords[:, None] - gt_coords[None]) ** 2).sum(axis=2)
        cost = distances_squared.copy()
        cost[distances_squared > MAX_LINK_UM**2] = 1e6
        pred_indices, gt_indices = linear_sum_assignment(cost)
        for pred_index, gt_index in zip(pred_indices, gt_indices):
            if cost[pred_index, gt_index] < 1e6:
                matches[pred_nodes[pred_index][0]] = gt_nodes[gt_index][0]
    return matches


def find_estimated_nodes(value):
    if isinstance(value, dict):
        if "estimated_number_of_nodes" in value:
            return value["estimated_number_of_nodes"]
        for child in value.values():
            result = find_estimated_nodes(child)
            if result is not None:
                return result
    return None


def evaluate_edges(graph, geff_path):
    gt_ids, gt_t, gt_z, gt_y, gt_x, gt_edges = read_geff(geff_path)
    matches = match_nodes(graph, gt_ids, gt_t, gt_z, gt_y, gt_x)
    gt_edge_set = {(int(source), int(target)) for source, target in gt_edges}
    gt_out, gt_in = defaultdict(set), defaultdict(set)
    for source, target in gt_edge_set:
        gt_out[source].add(target)
        gt_in[target].add(source)

    true_positives, false_positives = 0, 0
    matched_edges = set()
    for source, target in graph.edges:
        gt_source, gt_target = matches.get(source), matches.get(target)
        if gt_source is None or gt_target is None:
            continue
        if (gt_source, gt_target) in gt_edge_set:
            true_positives += 1
            matched_edges.add((gt_source, gt_target))
        elif (
            gt_target in gt_in and gt_source not in gt_in[gt_target]
        ) or (
            gt_source in gt_out and gt_target not in gt_out[gt_source]
        ):
            false_positives += 1

    false_negatives = len(gt_edge_set) - len(matched_edges)
    denominator = true_positives + false_positives + false_negatives
    jaccard = true_positives / denominator if denominator else 0.0
    metadata = json.loads((geff_path / "zarr.json").read_text())
    estimated_nodes = find_estimated_nodes(metadata)
    adjustment = 1 - 0.1 * (len(graph.nodes) - estimated_nodes) / estimated_nodes

    return {
        "edge_tp": true_positives,
        "edge_fp": false_positives,
        "edge_fn": false_negatives,
        "edge_jaccard": jaccard,
        "adjusted_edge_jaccard": max(0.0, jaccard * adjustment),
        "node_recall": len(matches) / len(gt_ids),
        "predicted_nodes": len(graph.nodes),
        "estimated_nodes": estimated_nodes,
    }

## Validation

In [7]:
validation_samples = [
    "44b6_0113de3b",
    "6bba_784a78c9",
    "6bba_05b6850b",
]

validation_results = {}
if RUN_VALIDATION:
    for sample in validation_samples:
        volume = open_volume(TRAIN_DIR / f"{sample}.zarr")
        graph = track_volume(volume)
        validation_results[sample] = evaluate_edges(
            graph, TRAIN_DIR / f"{sample}.geff"
        )

validation_results

{'44b6_0113de3b': {'edge_tp': 47,
  'edge_fp': 0,
  'edge_fn': 3,
  'edge_jaccard': 0.94,
  'adjusted_edge_jaccard': 0.9485477771306541,
  'node_recall': 1.0,
  'predicted_nodes': 23413,
  'estimated_nodes': 25755},
 '6bba_784a78c9': {'edge_tp': 1410,
  'edge_fp': 0,
  'edge_fn': 329,
  'edge_jaccard': 0.8108108108108109,
  'adjusted_edge_jaccard': 0.8281711766745365,
  'node_recall': 0.9071588366890381,
  'predicted_nodes': 10292,
  'estimated_nodes': 13096},
 '6bba_05b6850b': {'edge_tp': 633,
  'edge_fp': 0,
  'edge_fn': 212,
  'edge_jaccard': 0.749112426035503,
  'adjusted_edge_jaccard': 0.7612757887531181,
  'node_recall': 0.8257839721254355,
  'predicted_nodes': 5329,
  'estimated_nodes': 6362}}

## Submission Formatting

In [8]:
COLUMNS = [
    "id", "dataset", "row_type", "node_id", "t",
    "z", "y", "x", "source_id", "target_id",
]


def write_graph(writer, dataset, graph, row_id):
    for node_id, t, z, y, x in graph.nodes:
        writer.writerow([
            row_id, dataset, "node", node_id, t, z, y, x, -1, -1
        ])
        row_id += 1
    for source_id, target_id in graph.edges:
        writer.writerow([
            row_id, dataset, "edge", -1, -1, -1, -1, -1,
            source_id, target_id,
        ])
        row_id += 1
    return row_id

## Test Inference

In [9]:
row_id = 0
with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(COLUMNS)
    for index, sample in enumerate(test_samples, start=1):
        volume = open_volume(TEST_DIR / f"{sample}.zarr")
        graph = track_volume(volume)
        row_id = write_graph(writer, sample, graph, row_id)
        print(
            f"[{index}/{len(test_samples)}] {sample}: "
            f"{len(graph.nodes)} nodes, {len(graph.edges)} edges"
        )

OUTPUT_PATH, row_id

44b6_0113de3b 23413 21549
44b6_0b24845f 13769 9842
6bba_05b6850b 5329 4736
6bba_05db0fb1 36349 31841


(WindowsPath('submission.csv'), 146828)

## Submission Checks

In [10]:
datasets = set()
row_count = 0

with OUTPUT_PATH.open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames == COLUMNS
    for expected_id, row in enumerate(reader):
        assert int(row["id"]) == expected_id
        datasets.add(row["dataset"])
        if row["row_type"] == "node":
            assert row["source_id"] == row["target_id"] == "-1"
        else:
            assert row["row_type"] == "edge"
            assert all(row[column] == "-1" for column in ["node_id", "t", "z", "y", "x"])
        row_count += 1

assert datasets == set(test_samples)
assert row_count > 0

row_count, OUTPUT_PATH.stat().st_size

(146828, 7559853)